# N-Candidate Solver — Drawer Slides (N=1)

The simplest case: a **single-product family**. No Cartesian product — the solver evaluates each slide individually against requirements.

**Compare with:**
- `v2_hinge_n_candidate_demo.ipynb` — N=2 (concealed hinges)
- `v2_n_candidate_demo.ipynb` — N=3 (LED lighting)

In [2]:
import sys
from pathlib import Path

# Find project root
_project_root = Path.cwd()
while _project_root != _project_root.parent:
    if (_project_root / "sample-data").exists() and (_project_root / "engine_v2").exists():
        break
    _project_root = _project_root.parent
sys.path.insert(0, str(_project_root))
DATA_DIR = _project_root / "sample-data"

from engine_v2.core.solver_n import NCandidateSolver
from engine_v2.families.drawer_slide.config import SLIDE_N_CONFIG
from engine_v2.families.drawer_slide.loader import load_from_json
from engine_v2.families.drawer_slide.models import (
    ExtensionType, SlideCloseType, SlideMountType, SlideRequirements,
)

all_slides = load_from_json(DATA_DIR)
solver = NCandidateSolver(config=SLIDE_N_CONFIG, product_lists={"slide": all_slides})

print(f"Loaded {len(all_slides)} drawer slides")
print(f"Roles: {SLIDE_N_CONFIG.role_names}  (N={len(SLIDE_N_CONFIG.roles)})")
print(f"Rules: {len(SLIDE_N_CONFIG.rules)}")

Loaded 4 drawer slides
Roles: ['slide']  (N=1)
Rules: 8


## 1. Product Catalog

In [3]:
print(f"{'SKU':<16} {'Brand':<8} {'Series':<26} {'Length':>6} {'Load':>6} {'Extension':<16} {'Mount':<14} {'Close':<12} {'Price':>7}")
print("-" * 110)
for s in all_slides:
    print(f"{s.sku:<16} {s.brand:<8} {s.series:<26} {s.slide_length_mm:>4}mm {s.max_load_kg:>4.0f}kg {s.extension_type.value:<16} {s.mount_type.value:<14} {s.close_type.value:<12} ${s.price_usd:.2f}")

SKU              Brand    Series                     Length   Load Extension        Mount          Close          Price
--------------------------------------------------------------------------------------------------------------
563H5330B        Blum     TANDEM plus BLUMOTION       533mm   30kg full             undermount     soft_close   $45.00
DWD-XP-533       Grass    Dynapro                     500mm   40kg full             undermount     soft_close   $35.00
KV-8400-18       KV       8400                        450mm   45kg three_quarter    side_mount     standard     $8.00
KV-CM-450        KV       Center Mount                450mm   15kg three_quarter    center_mount   self_close   $6.00


## 2. Scenarios

In [4]:
# Scenario 1: Standard kitchen drawer
req = SlideRequirements(cabinet_depth_mm=550, drawer_weight_kg=15.0)
result = solver.solve_with_explanation(req)

print(f"Status: {result['status']}")
print(f"{result['message']}\n")

if result["status"] == "solved":
    rec = result["recommended"]
    print(f"Recommended: {rec['candidates']['slide']['sku']}")
    print(f"Price: ${rec['total_price_usd']:.2f}\n")
    print(f"Constraint trace ({len(rec['constraint_trace'])} rules):")
    for rule in rec["constraint_trace"]:
        icon = "PASS" if rule["passed"] else "FAIL"
        print(f"  [{icon}] {rule['rule']} {rule['name']}: {rule['detail']}")
    print(f"\n+ {len(result['alternatives'])} alternative(s)")

Status: solved
Found 4 valid configuration(s)

Recommended: KV-CM-450
Price: $6.00

Constraint trace (8 rules):
  [PASS] DS001 load_capacity: Drawer 15.0kg <= slide rating 15.0kg
  [PASS] DS002 cabinet_depth: Cabinet 550mm >= required 450mm (slide 450mm + 0mm clearance)
  [PASS] DS003 extension_type: No extension type preference — any type accepted
  [PASS] DS004 mount_type: No mount type preference — any type accepted
  [PASS] DS005 undermount_width_limit: Not an undermount slide or drawer width not specified — rule skipped
  [PASS] DS006 disconnect_feature: Disconnect not required
  [PASS] DS007 soft_close: Soft-close not requested
  [PASS] DS008 push_open: Push-open not requested

+ 3 alternative(s)


In [5]:
# Scenario 2: Heavy-duty
results = solver.solve(SlideRequirements(cabinet_depth_mm=550, drawer_weight_kg=42.0))
print(f"Found {len(results)} slide(s) for 42kg drawer:")
for config in results:
    s = config.candidates["slide"]
    print(f"  {s.sku} - {s.brand} {s.series}, {s.max_load_kg}kg rating")

Found 1 slide(s) for 42kg drawer:
  KV-8400-18 - KV 8400, 45.0kg rating


In [6]:
# Scenario 3: Impossible - cabinet too shallow
fail = solver.solve_with_explanation(SlideRequirements(cabinet_depth_mm=300, drawer_weight_kg=5.0))
print(f"Status: {fail['status']}")
print(f"{fail['message']}\n")
if fail.get("failed_rules"):
    for fr in fail["failed_rules"]:
        print(f"  [{fr['rule']}] {fr['name']}: {fr['detail']}")

Status: no_solution
No valid configuration exists for these requirements

  [DS002] cabinet_depth: Cabinet 300mm < required 543mm (slide 533mm + 10mm clearance)


## 3. Exhaustive Evaluation

With N=1, every slide is evaluated individually. No Cartesian product.

In [7]:
solver.config.early_termination = False
baseline = SlideRequirements(cabinet_depth_mm=550, drawer_weight_kg=15.0)

print(f"{'SKU':<16} {'Valid':>5}  Rule Results")
print("-" * 90)
for slide in all_slides:
    config = solver.evaluate({"slide": slide}, baseline)
    status = "YES" if config.valid else "NO"
    rules = "  ".join(f"{r.rule_id}:{'PASS' if r.passed else 'FAIL'}" for r in config.rule_results)
    print(f"{slide.sku:<16} {status:>5}  {rules}")

solver.config.early_termination = True
print(f"\nTotal: {len(all_slides)} slides x {len(SLIDE_N_CONFIG.rules)} rules = {len(all_slides) * len(SLIDE_N_CONFIG.rules)} evaluations")

SKU              Valid  Rule Results
------------------------------------------------------------------------------------------
563H5330B          YES  DS001:PASS  DS002:PASS  DS003:PASS  DS004:PASS  DS005:PASS  DS006:PASS  DS007:PASS  DS008:PASS
DWD-XP-533         YES  DS001:PASS  DS002:PASS  DS003:PASS  DS004:PASS  DS005:PASS  DS006:PASS  DS007:PASS  DS008:PASS
KV-8400-18         YES  DS001:PASS  DS002:PASS  DS003:PASS  DS004:PASS  DS005:PASS  DS006:PASS  DS007:PASS  DS008:PASS
KV-CM-450          YES  DS001:PASS  DS002:PASS  DS003:PASS  DS004:PASS  DS005:PASS  DS006:PASS  DS007:PASS  DS008:PASS

Total: 4 slides x 8 rules = 32 evaluations
